
# 03_import_kqapro.ipynb

Этот ноутбук нужен для **следующего внешнего слоя** твоего бенчмарка:  
мы берём **KQA Pro** как источник **сложных compositional / multi-hop KBQA-запросов**, вытаскиваем из него вопросы, SPARQL и KoPL-программы, автоматически оцениваем композиционную сложность и экспортируем **кандидатов на русскую реконструкцию**.

## Что делает ноутбук
1. Загружает **KQA Pro** через Hugging Face (`drt/kqa_pro`) или из локальных `train.json` / `val.json`.
2. Смотрит на структуру `question / sparql / program / answer / choices`.
3. Строит **словарь KoPL-функций** и производные признаки сложности.
4. Размечает записи по reasoning family:
   - `multi_hop`
   - `comparison_or_superlative`
   - `count`
   - `logical_set`
   - `temporal`
   - `qualifier`
   - `verification`
   - `filtering`
5. Отбирает **сложных кандидатов**, которые лучше подходят для твоего русского multi-hop benchmark-а, чем простые one-hop вопросы.
6. Экспортирует результаты в `jsonl/csv` для следующего шага — уже русской реконструкции.

## Важно
- В этом ноутбуке **нет перевода на русский**.
- В этом ноутбуке **нет dedup**.
- В этом ноутбуке **нет финального санитичека**.
- Это именно ноутбук уровня **import + structure mining + complex candidate extraction**.


In [7]:
%pip install -q datasets pandas pyarrow requests matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
from __future__ import annotations

import json
import math
import random
import re
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import pandas as pd
import requests
from datasets import load_dataset


In [9]:
# =========================
# CONFIG
# =========================

PROJECT_ROOT = Path(".").resolve()

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts_kqapro_stage2"
JSONL_DIR = ARTIFACTS_DIR / "jsonl"
CSV_DIR = ARTIFACTS_DIR / "csv"
REPORTS_DIR = ARTIFACTS_DIR / "reports"
CACHE_DIR = ARTIFACTS_DIR / "_cache_raw"

for p in [ARTIFACTS_DIR, JSONL_DIR, CSV_DIR, REPORTS_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Основной способ загрузки
USE_HF = True
HF_DATASET_NAME = "caobiao24/kqa_pro"
HF_CONFIG_NAME = "train_val"

# Файловый fallback через прямую загрузку json с Hugging Face
HF_RAW_BASE_URL = "https://huggingface.co/datasets/caobiao24/kqa_pro/resolve/main"
RAW_DOWNLOAD_TIMEOUT = 120
FORCE_REDOWNLOAD_RAW = False

# Локальный fallback: положи сюда train.json и val.json, если интернет / HF не работают
LOCAL_KQAPRO_DIR: Optional[Path] = None
# пример:
# LOCAL_KQAPRO_DIR = PROJECT_ROOT / "kqapro_dataset"

# Параметры отбора
RANDOM_SEED = 42
MIN_COMPLEXITY_SCORE = 4
STRICT_MIN_COMPLEXITY_SCORE = 6
MIN_PROGRAM_LENGTH = 4
MAX_PER_SIGNATURE = 30
TARGET_BALANCED_SHORTLIST = 600

random.seed(RANDOM_SEED)

ARTIFACTS_DIR


PosixPath('/Users/matvey/Desktop/kqapro/artifacts_kqapro_stage2')

In [10]:

# =========================
# БАЗОВЫЕ УТИЛИТЫ
# =========================

def normalize_spaces(text: Optional[str]) -> str:
    if text is None:
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def safe_str(x: Any) -> str:
    if x is None:
        return ""
    return normalize_spaces(str(x))


def save_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def maybe_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def preview_rows(rows: List[Dict[str, Any]], n: int = 3, keys: Optional[List[str]] = None) -> None:
    keys = keys or ["question", "answer", "sparql", "program"]
    for i, row in enumerate(rows[:n]):
        print("=" * 100)
        print(f"ROW #{i}")
        for k in keys:
            v = row.get(k)
            if isinstance(v, (dict, list)):
                print(f"{k}:")
                print(json.dumps(v, ensure_ascii=False, indent=2)[:2500])
            else:
                print(f"{k}: {v}")
        print()


def to_records(ds_split) -> List[Dict[str, Any]]:
    if ds_split is None:
        return []
    return [dict(x) for x in ds_split]


In [11]:
# =========================
# ЗАГРУЗКА KQA PRO
# =========================

def load_kqapro_hf() -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Пробуем стандартный datasets.load_dataset.
    Может не сработать на новых версиях datasets, если у датасета есть dataset script.
    """
    ds = load_dataset(HF_DATASET_NAME, HF_CONFIG_NAME)

    train_key = "train" if "train" in ds else None
    val_key = "validation" if "validation" in ds else ("val" if "val" in ds else None)

    if train_key is None or val_key is None:
        raise RuntimeError(f"Неожиданные split keys у HF dataset: {list(ds.keys())}")

    train_rows = to_records(ds[train_key])
    val_rows = to_records(ds[val_key])
    return train_rows, val_rows


def _download_to(url: str, dst: Path, timeout: int = RAW_DOWNLOAD_TIMEOUT) -> Path:
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + ".part")
    print(f"[download] {url}")
    with requests.get(url, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    tmp.replace(dst)
    return dst


def load_kqapro_from_hf_raw(base_url: str = HF_RAW_BASE_URL,
                            cache_dir: Path = CACHE_DIR,
                            force_redownload: bool = FORCE_REDOWNLOAD_RAW) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Файловый fallback: качаем train.json и val.json напрямую из repo на Hugging Face.
    Это обходит проблему 'Dataset scripts are no longer supported'.
    """
    train_path = cache_dir / "train.json"
    val_path = cache_dir / "val.json"

    files = {
        train_path: f"{base_url}/train.json",
        val_path: f"{base_url}/val.json",
    }

    for dst, url in files.items():
        if force_redownload or (not dst.exists()) or dst.stat().st_size == 0:
            _download_to(url, dst)
        else:
            print(f"[cache] using existing file: {dst}")

    train_rows = load_json(train_path)
    val_rows = load_json(val_path)
    return train_rows, val_rows


def load_kqapro_local(local_dir: Path) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    train_path = local_dir / "train.json"
    val_path = local_dir / "val.json"

    if not train_path.exists() or not val_path.exists():
        raise FileNotFoundError(
            f"Ожидались файлы {train_path} и {val_path}, но один из них не найден."
        )

    train_rows = load_json(train_path)
    val_rows = load_json(val_path)
    return train_rows, val_rows


train_raw: List[Dict[str, Any]] = []
val_raw: List[Dict[str, Any]] = []
load_method = None
errors = []

# 1) Пробуем datasets.load_dataset
if USE_HF:
    try:
        print("[load] trying Hugging Face via datasets.load_dataset ...")
        train_raw, val_raw = load_kqapro_hf()
        load_method = "hf_datasets"
        print("[load] HF datasets success")
    except Exception as e:
        errors.append(("hf_datasets", f"{type(e).__name__}: {e}"))
        print(f"[load] HF datasets failed: {type(e).__name__}: {e}")

# 2) Если не получилось, качаем raw json напрямую из HF repo
if load_method is None:
    try:
        print("[load] trying raw JSON download from Hugging Face repo ...")
        train_raw, val_raw = load_kqapro_from_hf_raw()
        load_method = "hf_raw_json"
        print("[load] raw JSON fallback success")
    except Exception as e:
        errors.append(("hf_raw_json", f"{type(e).__name__}: {e}"))
        print(f"[load] raw JSON fallback failed: {type(e).__name__}: {e}")

# 3) Последний fallback — локальные файлы
if load_method is None and LOCAL_KQAPRO_DIR is not None:
    try:
        print(f"[load] trying local fallback from {LOCAL_KQAPRO_DIR} ...")
        train_raw, val_raw = load_kqapro_local(LOCAL_KQAPRO_DIR)
        load_method = "local_json"
        print("[load] local fallback success")
    except Exception as e:
        errors.append(("local_json", f"{type(e).__name__}: {e}"))
        print(f"[load] local fallback failed: {type(e).__name__}: {e}")

if load_method is None:
    pretty_errors = "".join([f"  - {name}: {msg}" for name, msg in errors])
    raise RuntimeError(
        "Не удалось загрузить KQA Pro ни одним способом.\n"
        "Проверь интернет или положи локально train.json / val.json и укажи LOCAL_KQAPRO_DIR.\n"
        f"Ошибки:\n{pretty_errors}"
    )

print(f"[load] method used: {load_method}")
print(f"train_raw: {len(train_raw):,}")
print(f"val_raw:   {len(val_raw):,}")


[load] trying Hugging Face via datasets.load_dataset ...
[load] HF datasets failed: RuntimeError: Dataset scripts are no longer supported, but found kqa_pro.py
[load] trying raw JSON download from Hugging Face repo ...
[cache] using existing file: /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage2/_cache_raw/train.json
[download] https://huggingface.co/datasets/caobiao24/kqa_pro/resolve/main/val.json
[load] raw JSON fallback success
[load] method used: hf_raw_json
train_raw: 94,376
val_raw:   11,797


In [12]:

# Базовый sanity-check структуры
assert len(train_raw) > 0 and len(val_raw) > 0, "Пустой train/val"
required_train_keys = {"question", "sparql", "program", "choices", "answer"}
missing = required_train_keys - set(train_raw[0].keys())
assert not missing, f"В train первой записи не хватает ключей: {missing}"

preview_rows(train_raw, n=2)


ROW #0
question: Which town has a TOID of 4000000074573917 and has an OS grid reference of SP8778?
answer: Kettering
sparql: SELECT DISTINCT ?e WHERE { ?e <pred:instance_of> ?c . ?c <pred:name> "town" . ?e <TOID> ?pv . ?pv <pred:value> "4000000074573917" . ?e <OS_grid_reference> ?pv_1 . ?pv_1 <pred:value> "SP8778" .  }
program:
[
  {
    "function": "FindAll",
    "dependencies": [],
    "inputs": []
  },
  {
    "function": "FilterStr",
    "dependencies": [
      0
    ],
    "inputs": [
      "TOID",
      "4000000074573917"
    ]
  },
  {
    "function": "FilterConcept",
    "dependencies": [
      1
    ],
    "inputs": [
      "town"
    ]
  },
  {
    "function": "FindAll",
    "dependencies": [],
    "inputs": []
  },
  {
    "function": "FilterStr",
    "dependencies": [
      3
    ],
    "inputs": [
      "OS grid reference",
      "SP8778"
    ]
  },
  {
    "function": "FilterConcept",
    "dependencies": [
      4
    ],
    "inputs": [
      "town"
    ]
  },
  {
    "fu


## Словарь функций KoPL

KQA Pro даёт для каждого вопроса **KoPL program** и **SPARQL**.  
Чтобы отобрать именно **сложные** примеры для твоего русского бенча, полезно сначала посмотреть, какие функции реально встречаются в программах и насколько они длинные.


In [13]:

def extract_program_functions(program: Any) -> List[str]:
    out = []
    for step in maybe_list(program):
        if isinstance(step, dict):
            func = step.get("function")
            if func is not None:
                out.append(str(func))
    return out


all_functions = Counter()
program_lengths = []
for row in train_raw + val_raw:
    funcs = extract_program_functions(row.get("program"))
    all_functions.update(funcs)
    program_lengths.append(len(funcs))

func_df = pd.DataFrame(
    [{"function": fn, "count": cnt} for fn, cnt in all_functions.most_common()]
)
func_df.head(50)


,function,count
0,Find,144515
1,FilterConcept,67604
2,Relate,56631
3,FindAll,34132
4,FilterStr,28569
5,And,28539
6,QueryAttr,25022
7,QueryRelation,15662
8,SelectBetween,14851
9,What,12632


In [14]:

func_df.to_csv(CSV_DIR / "kqapro_function_vocab.csv", index=False)
print("saved:", CSV_DIR / "kqapro_function_vocab.csv")
print("num unique functions:", len(func_df))
print("program length min/max:", min(program_lengths), max(program_lengths))
print("program length mean:", round(sum(program_lengths) / len(program_lengths), 2))


saved: /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage2/csv/kqapro_function_vocab.csv
num unique functions: 27
program length min/max: 2 15
program length mean: 4.79



## Автоматическая разметка reasoning family

У KQA Pro нет готового русского слоя, зато есть очень сильная структура:  
`question + answer + SPARQL + KoPL program`.

Ниже мы строим **эвристическую таксономию** по KoPL-функциям.  
Это не финальная “истина”, но для отбора сложных кандидатов работает хорошо:
- `multi_hop`
- `count`
- `comparison_or_superlative`
- `logical_set`
- `verification`
- `qualifier`
- `temporal`
- `filtering`
- `high_compositional_depth`


In [15]:

ADVANCED_REASONING_TAGS = {
    "multi_hop",
    "count",
    "comparison_or_superlative",
    "logical_set",
    "verification",
    "qualifier",
    "temporal",
    "high_compositional_depth",
}

def looks_like_year_or_date(text: str) -> bool:
    text = safe_str(text)
    if re.search(r"\b\d{4}\b", text):
        return True
    if re.search(r"\d{4}[-/]\d{1,2}[-/]\d{1,2}", text):
        return True
    if re.search(r"\b(before|after|during|between|year|date|month|day)\b", text.lower()):
        return True
    return False


def infer_reasoning_tags(program: Any, question: str = "", sparql: str = "") -> List[str]:
    funcs = extract_program_functions(program)
    tags = set()

    relate_count = sum(1 for f in funcs if "Relate" in f)
    find_count = sum(1 for f in funcs if f.startswith("Find"))
    filter_count = sum(1 for f in funcs if "Filter" in f)
    qualifier_count = sum(1 for f in funcs if ("Qualifier" in f or f.startswith("QFilter")))
    temporal_fn = sum(1 for f in funcs if ("Year" in f or "Date" in f or f.startswith("QFilterYear") or f.startswith("QFilterDate")))

    if relate_count >= 2 or (relate_count >= 1 and find_count >= 2):
        tags.add("multi_hop")

    if any("Count" in f for f in funcs):
        tags.add("count")

    if any(("Select" in f) or ("Among" in f) or ("Between" in f) for f in funcs):
        tags.add("comparison_or_superlative")

    if any(f in {"And", "Or"} for f in funcs):
        tags.add("logical_set")

    if any("Verify" in f for f in funcs):
        tags.add("verification")

    if qualifier_count > 0:
        tags.add("qualifier")

    if temporal_fn > 0 or looks_like_year_or_date(question) or looks_like_year_or_date(sparql):
        tags.add("temporal")

    if filter_count > 0:
        tags.add("filtering")

    if len(funcs) >= 6:
        tags.add("high_compositional_depth")

    if not tags:
        tags.add("other_structured")

    return sorted(tags)


def estimate_hop_count(program: Any) -> int:
    funcs = extract_program_functions(program)
    hop_count = sum(1 for f in funcs if "Relate" in f)
    if hop_count == 0 and len(funcs) >= 4:
        return 1
    return hop_count


def complexity_score(program: Any, question: str = "", sparql: str = "") -> int:
    funcs = extract_program_functions(program)
    tags = infer_reasoning_tags(program, question, sparql)
    score = 0

    score += min(len(funcs), 10)

    bonus_map = {
        "multi_hop": 3,
        "count": 2,
        "comparison_or_superlative": 3,
        "logical_set": 3,
        "verification": 2,
        "qualifier": 2,
        "temporal": 1,
        "filtering": 1,
        "high_compositional_depth": 3,
    }
    for t in tags:
        score += bonus_map.get(t, 0)

    return score


def primary_reasoning_family(tags: List[str]) -> str:
    priority = [
        "multi_hop",
        "logical_set",
        "comparison_or_superlative",
        "count",
        "verification",
        "qualifier",
        "temporal",
        "filtering",
        "high_compositional_depth",
        "other_structured",
    ]
    for p in priority:
        if p in tags:
            return p
    return "other_structured"


In [16]:

tag_counter = Counter()
for row in train_raw:
    for t in infer_reasoning_tags(row["program"], row.get("question", ""), row.get("sparql", "")):
        tag_counter[t] += 1

pd.DataFrame(
    [{"reasoning_tag": k, "count": v} for k, v in tag_counter.most_common()]
)


,reasoning_tag,count
0,filtering,56041
1,temporal,32263
2,high_compositional_depth,30233
3,logical_set,26524
4,multi_hop,25732
5,qualifier,21629
6,comparison_or_superlative,17535
7,verification,11653
8,count,10946
9,other_structured,5828



## Нормализация в единый формат

Здесь мы приводим KQA Pro к твоему benchmark-oriented schema.  
Важно: это **не финальный формат merge-а**, а **stage-2 import / template mining format**.

Особенно важные поля:
- `question_en_original`
- `program`
- `program_functions`
- `program_signature`
- `sparql`
- `gold_answers`
- `reasoning_tags`
- `reconstruction_priority_score`
- `reconstruction_prompt`


In [17]:

def serialize_program_signature(funcs: List[str]) -> str:
    return " -> ".join(funcs)


def build_reconstruction_prompt(question_en: str, answer: str, program: Any, sparql: str, tags: List[str]) -> str:
    program_txt = json.dumps(program, ensure_ascii=False)
    tags_txt = ", ".join(tags)
    return (
        "Сгенерируй естественный запрос на русском языке для benchmark-а, "
        "сохранив смысл исходного английского вопроса, все ограничения, тип reasoning и ожидаемый ответ. "
        "Не упрощай запрос до однохопового факта. "
        f"Original English question: {question_en}\n"
        f"Gold answer: {answer}\n"
        f"Reasoning tags: {tags_txt}\n"
        f"KoPL program: {program_txt}\n"
        f"SPARQL: {sparql}"
    )


def normalize_kqapro_rows(rows: List[Dict[str, Any]], split_name: str) -> List[Dict[str, Any]]:
    out = []

    for i, row in enumerate(rows):
        question = safe_str(row.get("question"))
        sparql = safe_str(row.get("sparql"))
        answer = row.get("answer")
        program = maybe_list(row.get("program"))
        choices = maybe_list(row.get("choices"))

        funcs = extract_program_functions(program)
        tags = infer_reasoning_tags(program, question=question, sparql=sparql)
        score = complexity_score(program, question=question, sparql=sparql)

        answer_text = safe_str(answer)

        norm = {
            "benchmark_id": f"kqapro_{split_name}_{i:06d}",
            "source_dataset": "kqa_pro",
            "source_split": split_name,
            "import_mode": "template_transfer",
            "role": "template_transfer_candidate",
            "question_ru": None,
            "question_en_original": question,
            "domain": None,
            "reasoning_tags": tags,
            "primary_reasoning_family": primary_reasoning_family(tags),
            "hop_count_estimate": estimate_hop_count(program),
            "program_length": len(funcs),
            "program_functions": funcs,
            "program_signature": serialize_program_signature(funcs),
            "formal_query": sparql,
            "program": program,
            "gold_answers": [{"text": answer_text}] if answer_text else [],
            "gold_qids": [],
            "answer_text": answer_text,
            "choices": choices,
            "choice_count": len(choices),
            "expected_cardinality": 1 if answer_text else None,
            "is_answerable": bool(answer_text),
            "reconstruction_priority_score": score,
            "reconstruction_prompt": build_reconstruction_prompt(
                question_en=question,
                answer=answer_text,
                program=program,
                sparql=sparql,
                tags=tags,
            ),
            "notes": "",
        }
        out.append(norm)

    return out


train_norm = normalize_kqapro_rows(train_raw, "train")
val_norm = normalize_kqapro_rows(val_raw, "val")

print(f"train_norm: {len(train_norm):,}")
print(f"val_norm:   {len(val_norm):,}")
print(f"all_norm:   {len(train_norm) + len(val_norm):,}")


train_norm: 94,376
val_norm:   11,797
all_norm:   106,173


In [18]:

all_norm = train_norm + val_norm
norm_df = pd.DataFrame(all_norm)

summary_basic = {
    "num_rows": len(norm_df),
    "num_train": len(train_norm),
    "num_val": len(val_norm),
    "mean_program_length": float(norm_df["program_length"].mean()),
    "median_program_length": float(norm_df["program_length"].median()),
    "mean_priority_score": float(norm_df["reconstruction_priority_score"].mean()),
}

summary_basic


{'num_rows': 106173,
 'num_train': 94376,
 'num_val': 11797,
 'mean_program_length': 4.7866312527667105,
 'median_program_length': 4.0,
 'mean_priority_score': 9.831576766221167}

In [19]:

family_df = (
    norm_df.groupby("primary_reasoning_family")
    .agg(
        count=("benchmark_id", "count"),
        mean_program_length=("program_length", "mean"),
        mean_priority_score=("reconstruction_priority_score", "mean"),
    )
    .reset_index()
    .sort_values(["count", "mean_priority_score"], ascending=[False, False])
)

family_df


,primary_reasoning_family,count,mean_program_length,mean_priority_score
4,multi_hop,29047,7.030984,17.937274
6,qualifier,17488,3.682925,6.672061
0,comparison_or_superlative,13624,3.608999,7.339034
8,verification,11507,4.443382,8.195968
7,temporal,10325,3.611913,5.266828
2,filtering,10258,4.225288,5.540846
5,other_structured,6507,2.541571,2.541571
1,count,5041,4.558024,8.649474
3,logical_set,2376,8.049242,17.039983


In [20]:

tag_combo_df = (
    norm_df.assign(reasoning_combo=norm_df["reasoning_tags"].apply(lambda x: " | ".join(x)))
    .groupby("reasoning_combo")
    .agg(
        count=("benchmark_id", "count"),
        mean_program_length=("program_length", "mean"),
        mean_priority_score=("reconstruction_priority_score", "mean"),
    )
    .reset_index()
    .sort_values(["count", "mean_priority_score"], ascending=[False, False])
)

tag_combo_df.head(30)


,reasoning_combo,count,mean_program_length,mean_priority_score
24,filtering,9179,4.016668,5.016668
64,other_structured,6507,2.541571,2.541571
65,qualifier,6301,2.672750,4.672750
0,comparison_or_superlative,5860,3.000000,6.000000
49,filtering | qualifier,5585,4.241898,7.241898
1,comparison_or_superlative | filtering,5325,4.090516,8.090516
51,filtering | temporal,5194,4.029842,6.029842
67,temporal,4739,2.956320,3.956320
50,filtering | qualifier | temporal,4373,4.310313,8.310313
5,comparison_or_superlative | high_compositional...,4318,6.689208,18.689208



## Отбор сложных кандидатов

Здесь мы делаем то, чего не хватало в RuBQ:
мы **не пытаемся взять всё подряд**, а жёстче фильтруем только те примеры, которые:
- выглядят существенно сложнее типичного one-hop KBQA,
- имеют длиннее и богаче KoPL-программу,
- содержат multi-hop / count / set / comparison / qualifier / temporal reasoning.

Это не финальный gold-benchmark, а **сырьё для русской реконструкции**.


In [21]:

def is_simple_kbqa_candidate(row: Dict[str, Any]) -> bool:
    funcs = row["program_functions"]
    tags = set(row["reasoning_tags"])
    score = row["reconstruction_priority_score"]

    if len(funcs) <= 3 and not (tags & ADVANCED_REASONING_TAGS):
        return True

    if len(funcs) < MIN_PROGRAM_LENGTH and score < MIN_COMPLEXITY_SCORE:
        return True

    if (
        len(funcs) <= 4
        and "multi_hop" not in tags
        and "logical_set" not in tags
        and "comparison_or_superlative" not in tags
        and "count" not in tags
        and "verification" not in tags
        and "qualifier" not in tags
        and row["hop_count_estimate"] <= 1
    ):
        return True

    return False


def is_complex_candidate(row: Dict[str, Any]) -> bool:
    if is_simple_kbqa_candidate(row):
        return False

    tags = set(row["reasoning_tags"])
    score = row["reconstruction_priority_score"]

    if score >= STRICT_MIN_COMPLEXITY_SCORE:
        return True

    if row["program_length"] >= MIN_PROGRAM_LENGTH and (tags & ADVANCED_REASONING_TAGS):
        return True

    return False


complex_candidates = [r for r in all_norm if is_complex_candidate(r)]
simple_rejected = [r for r in all_norm if not is_complex_candidate(r)]

print(f"complex_candidates: {len(complex_candidates):,}")
print(f"simple_rejected:    {len(simple_rejected):,}")
print(f"keep ratio:         {len(complex_candidates) / max(len(all_norm), 1):.2%}")


complex_candidates: 72,792
simple_rejected:    33,381
keep ratio:         68.56%


In [22]:

complex_df = pd.DataFrame(complex_candidates)

complex_family_df = (
    complex_df.groupby("primary_reasoning_family")
    .agg(
        count=("benchmark_id", "count"),
        mean_program_length=("program_length", "mean"),
        mean_priority_score=("reconstruction_priority_score", "mean"),
    )
    .reset_index()
    .sort_values(["count", "mean_priority_score"], ascending=[False, False])
)

complex_family_df


,primary_reasoning_family,count,mean_program_length,mean_priority_score
4,multi_hop,29047,7.030984,17.937274
0,comparison_or_superlative,13624,3.608999,7.339034
5,qualifier,10688,4.357036,7.928799
7,verification,9675,4.716693,8.801137
1,count,5041,4.558024,8.649474
3,logical_set,2376,8.049242,17.039983
2,filtering,1515,5.712211,8.848845
6,temporal,826,5.474576,8.898305



## Balanced shortlist

Полезно не просто взять top-N по score, а построить **сбалансированный shortlist**:
- ограничить однообразные program signatures,
- не дать одному reasoning family “съесть” всё,
- получить удобную выборку для ручного просмотра и последующей русской реконструкции.


In [23]:

def group_key(row: Dict[str, Any]) -> str:
    return f"{row['primary_reasoning_family']} || {row['program_signature']}"


grouped = defaultdict(list)
for row in complex_candidates:
    grouped[group_key(row)].append(row)

for g in grouped:
    grouped[g].sort(
        key=lambda r: (
            -r["reconstruction_priority_score"],
            -r["program_length"],
            -len(r["question_en_original"]),
            r["benchmark_id"],
        )
    )

signature_capped = []
for g, rows in grouped.items():
    signature_capped.extend(rows[:MAX_PER_SIGNATURE])

family_buckets = defaultdict(list)
for row in signature_capped:
    family_buckets[row["primary_reasoning_family"]].append(row)

for fam in family_buckets:
    family_buckets[fam].sort(
        key=lambda r: (
            -r["reconstruction_priority_score"],
            -r["program_length"],
            -len(r["question_en_original"]),
            r["benchmark_id"],
        )
    )

families = sorted(family_buckets.keys())
if not families:
    raise RuntimeError("После фильтрации нет ни одного кандидата")

quota = math.ceil(TARGET_BALANCED_SHORTLIST / len(families))

balanced_shortlist = []
for fam in families:
    balanced_shortlist.extend(family_buckets[fam][:quota])

balanced_shortlist = sorted(
    balanced_shortlist,
    key=lambda r: (
        -r["reconstruction_priority_score"],
        -r["program_length"],
        r["primary_reasoning_family"],
        r["benchmark_id"],
    )
)[:TARGET_BALANCED_SHORTLIST]

print("families:", families)
print("quota per family:", quota)
print("balanced_shortlist:", len(balanced_shortlist))


families: ['comparison_or_superlative', 'count', 'filtering', 'logical_set', 'multi_hop', 'qualifier', 'temporal', 'verification']
quota per family: 75
balanced_shortlist: 600


In [24]:

shortlist_df = pd.DataFrame(balanced_shortlist)

shortlist_family_df = (
    shortlist_df.groupby("primary_reasoning_family")
    .agg(
        count=("benchmark_id", "count"),
        mean_program_length=("program_length", "mean"),
        mean_priority_score=("reconstruction_priority_score", "mean"),
    )
    .reset_index()
    .sort_values(["count", "mean_priority_score"], ascending=[False, False])
)

shortlist_family_df


,primary_reasoning_family,count,mean_program_length,mean_priority_score
4,multi_hop,75,10.080000,23.720000
3,logical_set,75,9.000000,19.000000
1,count,75,7.000000,15.506667
7,verification,75,6.346667,15.280000
5,qualifier,75,7.000000,14.000000
6,temporal,75,6.000000,11.000000
0,comparison_or_superlative,75,5.000000,10.000000
2,filtering,75,5.800000,9.200000


In [25]:

signature_df = (
    shortlist_df.groupby("program_signature")
    .agg(
        count=("benchmark_id", "count"),
        family=("primary_reasoning_family", "first"),
        mean_program_length=("program_length", "mean"),
        mean_priority_score=("reconstruction_priority_score", "mean"),
    )
    .reset_index()
    .sort_values(["count", "mean_priority_score"], ascending=[False, False])
)

signature_df.head(30)


,program_signature,count,family,mean_program_length,mean_priority_score
95,FindAll -> FilterStr -> FilterConcept -> Relat...,60,temporal,6.0,10.500000
74,FindAll -> FilterNum -> FilterConcept -> Relat...,32,temporal,6.0,10.062500
114,FindAll -> FilterStr -> QFilterStr -> FilterCo...,30,count,7.0,15.166667
64,FindAll -> FilterDate -> FilterConcept -> Rela...,30,temporal,6.0,11.000000
99,FindAll -> FilterStr -> FilterConcept -> Relat...,21,qualifier,7.0,14.000000
84,FindAll -> FilterNum -> QFilterYear -> FilterC...,16,count,7.0,16.000000
96,FindAll -> FilterStr -> FilterConcept -> Relat...,15,qualifier,7.0,14.000000
98,FindAll -> FilterStr -> FilterConcept -> Relat...,15,qualifier,7.0,14.000000
54,Find -> Relate -> Find -> And -> Relate -> QFi...,14,multi_hop,9.0,24.000000
125,FindAll -> FilterYear -> FilterConcept -> Rela...,13,temporal,6.0,11.000000


In [26]:

preview_cols = [
    "benchmark_id",
    "primary_reasoning_family",
    "reasoning_tags",
    "program_length",
    "hop_count_estimate",
    "reconstruction_priority_score",
    "question_en_original",
    "answer_text",
]

shortlist_df[preview_cols].head(25)


,benchmark_id,primary_reasoning_family,reasoning_tags,program_length,hop_count_estimate,reconstruction_priority_score,question_en_original,answer_text
0,kqapro_train_056652,multi_hop,"[count, filtering, high_compositional_depth, l...",10,3,25,How many hip hop music renditions originate in...,11
1,kqapro_train_072254,multi_hop,"[count, filtering, high_compositional_depth, l...",10,3,25,How many musicals received a theater award tha...,3
2,kqapro_train_019201,multi_hop,"[count, filtering, high_compositional_depth, l...",10,3,24,"How many counties of Ireland, which is an admi...",18
3,kqapro_train_019784,multi_hop,"[count, filtering, high_compositional_depth, l...",10,3,24,How many video games have been produced from t...,2
4,kqapro_train_022718,multi_hop,"[count, filtering, high_compositional_depth, l...",10,3,24,How many institutions of higher education rece...,2
5,kqapro_train_067956,multi_hop,"[count, filtering, high_compositional_depth, l...",10,3,24,How many Alabama counties are located in the s...,10
6,kqapro_train_089365,multi_hop,"[count, filtering, high_compositional_depth, l...",10,3,24,How many independent record labels belong to a...,1
7,kqapro_train_000591,multi_hop,"[filtering, high_compositional_depth, logical_...",9,2,24,Was the inception time of an association footb...,yes
8,kqapro_train_002593,multi_hop,"[filtering, high_compositional_depth, logical_...",9,2,24,Does the person who is a cast member of Aliens...,yes
9,kqapro_train_002599,multi_hop,"[filtering, high_compositional_depth, logical_...",9,2,24,"Was Shelley Duvalls, a cast member of The Shin...",yes


In [27]:

# =========================
# ЭКСПОРТ
# =========================

save_jsonl(JSONL_DIR / "kqapro_all_normalized.jsonl", all_norm)
save_jsonl(JSONL_DIR / "kqapro_complex_candidates.jsonl", complex_candidates)
save_jsonl(JSONL_DIR / "kqapro_balanced_shortlist.jsonl", balanced_shortlist)

norm_df.to_csv(CSV_DIR / "kqapro_all_normalized.csv", index=False)
complex_df.to_csv(CSV_DIR / "kqapro_complex_candidates.csv", index=False)
shortlist_df.to_csv(CSV_DIR / "kqapro_balanced_shortlist.csv", index=False)

family_df.to_csv(CSV_DIR / "kqapro_family_summary_all.csv", index=False)
complex_family_df.to_csv(CSV_DIR / "kqapro_family_summary_complex.csv", index=False)
shortlist_family_df.to_csv(CSV_DIR / "kqapro_family_summary_shortlist.csv", index=False)
signature_df.to_csv(CSV_DIR / "kqapro_signature_summary_shortlist.csv", index=False)

summary_json = {
    "source_dataset": "kqa_pro",
    "num_all_normalized": len(all_norm),
    "num_complex_candidates": len(complex_candidates),
    "num_balanced_shortlist": len(balanced_shortlist),
    "mean_program_length_all": float(norm_df["program_length"].mean()),
    "mean_program_length_complex": float(complex_df["program_length"].mean()) if len(complex_df) else 0.0,
    "mean_program_length_shortlist": float(shortlist_df["program_length"].mean()) if len(shortlist_df) else 0.0,
    "mean_priority_score_all": float(norm_df["reconstruction_priority_score"].mean()),
    "mean_priority_score_complex": float(complex_df["reconstruction_priority_score"].mean()) if len(complex_df) else 0.0,
    "mean_priority_score_shortlist": float(shortlist_df["reconstruction_priority_score"].mean()) if len(shortlist_df) else 0.0,
    "families_in_shortlist": shortlist_family_df.to_dict(orient="records"),
    "artifacts_dir": str(ARTIFACTS_DIR),
}

with open(REPORTS_DIR / "summary_kqapro_stage2.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)

print("saved JSONL:", [p.name for p in JSONL_DIR.iterdir()])
print("saved CSV:", [p.name for p in CSV_DIR.iterdir()])
print("saved REPORT:", REPORTS_DIR / "summary_kqapro_stage2.json")


saved JSONL: ['kqapro_complex_candidates.jsonl', 'kqapro_balanced_shortlist.jsonl', 'kqapro_all_normalized.jsonl']
saved CSV: ['kqapro_family_summary_complex.csv', 'kqapro_balanced_shortlist.csv', 'kqapro_function_vocab.csv', 'kqapro_family_summary_all.csv', 'kqapro_all_normalized.csv', 'kqapro_signature_summary_shortlist.csv', 'kqapro_family_summary_shortlist.csv', 'kqapro_complex_candidates.csv']
saved REPORT: /Users/matvey/Desktop/kqapro/artifacts_kqapro_stage2/reports/summary_kqapro_stage2.json



## Как интерпретировать результаты

Главные файлы:
- `kqapro_all_normalized.jsonl`  
  полный импортированный слой KQA Pro в твоём формате
- `kqapro_complex_candidates.jsonl`  
  только более сложные кандидаты
- `kqapro_balanced_shortlist.jsonl`  
  уже удобная **сбалансированная выборка** для ручного просмотра и следующего этапа
